# Dataset Analysis — Traffic Flow Prediction
**Datasets:** PeMS04, PeMS07, PeMS08, NYCTaxi

This notebook performs exploratory data analysis (EDA) on all four datasets to understand their structure, distributions, and temporal/spatial patterns before model training.

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Optional: for reading .npz PeMS files
# pip install h5py

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

## 2. Load Datasets

> **Note:** PeMS04/07/08 are stored as `.npz` files with shape `(T, N, F)` where T=timesteps, N=nodes/sensors, F=features (flow, occupancy, speed). NYCTaxi is typically a CSV.

In [ ]:
# ─── Adjust these paths to where you've saved the datasets ───
DATA_DIR = './data'

DATASETS = {
    'PeMS04': os.path.join(DATA_DIR, 'PEMS04/pems04.npz'),
    'PeMS07': os.path.join(DATA_DIR, 'PEMS07/pems07.npz'),
    'PeMS08': os.path.join(DATA_DIR, 'PEMS08/pems08.npz'),
    'NYCTaxi': os.path.join(DATA_DIR, 'NYCTaxi/nyc_taxi.csv'),
}

pems_data = {}
for name, path in DATASETS.items():
    if name != 'NYCTaxi':
        try:
            raw = np.load(path, allow_pickle=True)
            pems_data[name] = raw['data']  # shape: (T, N, F)
            print(f'{name} loaded — shape: {pems_data[name].shape}')
        except FileNotFoundError:
            print(f'[WARNING] {name} not found at {path}')

# NYCTaxi
try:
    nyc_df = pd.read_csv(DATASETS['NYCTaxi'], parse_dates=True)
    print(f'NYCTaxi loaded — shape: {nyc_df.shape}')
    print(nyc_df.head())
except FileNotFoundError:
    print('[WARNING] NYCTaxi CSV not found.')

## 3. Basic Statistics — PeMS Datasets

In [ ]:
feature_names = ['Flow', 'Occupancy', 'Speed']

for name, data in pems_data.items():
    T, N, F = data.shape
    print(f'\n=== {name} ===')
    print(f'  Timesteps (T): {T}')
    print(f'  Sensors   (N): {N}')
    print(f'  Features  (F): {F}')
    print(f'  Duration     : {T * 5 / 60:.1f} hours ({T * 5 / 1440:.1f} days) @ 5-min intervals')
    for f_idx in range(min(F, len(feature_names))):
        feat = data[:, :, f_idx].flatten()
        print(f'  {feature_names[f_idx]}: mean={feat.mean():.2f}, std={feat.std():.2f}, min={feat.min():.2f}, max={feat.max():.2f}')

## 4. Missing Value Analysis

In [ ]:
print('=== Missing Value Analysis ===\n')
for name, data in pems_data.items():
    total = data.size
    missing = np.sum(np.isnan(data))
    zeros = np.sum(data[:, :, 0] == 0)  # zero-flow sensors
    print(f'{name}:')
    print(f'  NaN values : {missing} ({100*missing/total:.3f}%)')
    print(f'  Zero flows : {zeros} ({100*zeros/(data.shape[0]*data.shape[1]):.2f}% of sensor-timesteps)')

# NYCTaxi
try:
    print('\nNYCTaxi:')
    print(nyc_df.isnull().sum())
except:
    pass

## 5. Temporal Patterns — Traffic Flow Over Time

In [ ]:
fig, axes = plt.subplots(len(pems_data), 1, figsize=(16, 4 * len(pems_data)))
if len(pems_data) == 1:
    axes = [axes]

for ax, (name, data) in zip(axes, pems_data.items()):
    # Average flow across all sensors over time
    avg_flow = data[:, :, 0].mean(axis=1)  # (T,)
    ax.plot(avg_flow, linewidth=0.8, color='steelblue', alpha=0.85)
    ax.set_title(f'{name} — Average Traffic Flow Over Time', fontsize=13, fontweight='bold')
    ax.set_xlabel('Timestep (5-min intervals)')
    ax.set_ylabel('Avg Flow')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('temporal_flow.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Daily & Weekly Patterns (Periodicity)

In [ ]:
# 5-min intervals → 288 steps/day
STEPS_PER_DAY = 288

for name, data in pems_data.items():
    T = data.shape[0]
    flow = data[:, :, 0].mean(axis=1)  # avg across sensors
    n_days = T // STEPS_PER_DAY
    flow_days = flow[:n_days * STEPS_PER_DAY].reshape(n_days, STEPS_PER_DAY)

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))

    # Average day profile
    mean_day = flow_days.mean(axis=0)
    std_day = flow_days.std(axis=0)
    time_labels = [f'{h:02d}:{m:02d}' for h in range(24) for m in range(0, 60, 5)]
    axes[0].plot(mean_day, color='steelblue', lw=2)
    axes[0].fill_between(range(STEPS_PER_DAY), mean_day - std_day, mean_day + std_day, alpha=0.2)
    axes[0].set_xticks(range(0, STEPS_PER_DAY, 24))
    axes[0].set_xticklabels([time_labels[i] for i in range(0, STEPS_PER_DAY, 24)], rotation=45)
    axes[0].set_title(f'{name} — Average Daily Pattern')
    axes[0].set_xlabel('Time of Day')
    axes[0].set_ylabel('Avg Flow')

    # Heatmap: day × time-of-day
    im = axes[1].imshow(flow_days[:min(14, n_days)], aspect='auto', cmap='YlOrRd')
    axes[1].set_title(f'{name} — Flow Heatmap (first 2 weeks)')
    axes[1].set_xlabel('Time of Day (5-min)')
    axes[1].set_ylabel('Day')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    plt.savefig(f'{name.lower()}_daily_pattern.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Spatial Distribution — Sensor-Level Flow Statistics

In [ ]:
for name, data in pems_data.items():
    sensor_mean_flow = data[:, :, 0].mean(axis=0)  # (N,)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(sensor_mean_flow, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].set_title(f'{name} — Distribution of Mean Flow per Sensor')
    axes[0].set_xlabel('Mean Flow')
    axes[0].set_ylabel('Count')

    axes[1].bar(range(len(sensor_mean_flow)), np.sort(sensor_mean_flow)[::-1],
                color='coral', edgecolor='none', width=1.0)
    axes[1].set_title(f'{name} — Sensors Ranked by Mean Flow')
    axes[1].set_xlabel('Sensor Rank')
    axes[1].set_ylabel('Mean Flow')

    plt.tight_layout()
    plt.savefig(f'{name.lower()}_spatial_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Feature Correlation (Flow / Occupancy / Speed)

In [ ]:
for name, data in pems_data.items():
    T, N, F = data.shape
    if F < 2:
        print(f'{name}: Only 1 feature, skipping correlation.')
        continue

    # Sample 5000 sensor-timestep pairs
    sample_idx = np.random.choice(T * N, size=min(5000, T * N), replace=False)
    flat = data.reshape(-1, F)[sample_idx]
    df_corr = pd.DataFrame(flat[:, :min(F, 3)], columns=feature_names[:min(F, 3)])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    corr = df_corr.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], vmin=-1, vmax=1)
    axes[0].set_title(f'{name} — Feature Correlation Matrix')

    if F >= 2:
        axes[1].scatter(df_corr.iloc[:, 0], df_corr.iloc[:, 1], alpha=0.3, s=5, color='steelblue')
        axes[1].set_xlabel(feature_names[0])
        axes[1].set_ylabel(feature_names[1])
        axes[1].set_title(f'{name} — {feature_names[0]} vs {feature_names[1]}')

    plt.tight_layout()
    plt.savefig(f'{name.lower()}_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. NYCTaxi — Additional EDA

In [ ]:
try:
    print(nyc_df.dtypes)
    print('\nFirst rows:')
    display(nyc_df.head())

    # Detect datetime column
    dt_cols = [c for c in nyc_df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if dt_cols:
        nyc_df[dt_cols[0]] = pd.to_datetime(nyc_df[dt_cols[0]])
        nyc_df = nyc_df.set_index(dt_cols[0]).sort_index()

        # Resample hourly if needed
        numeric_cols = nyc_df.select_dtypes(include=[np.number]).columns
        nyc_hourly = nyc_df[numeric_cols].resample('1H').mean()

        fig, ax = plt.subplots(figsize=(16, 4))
        nyc_hourly.iloc[:, 0].plot(ax=ax, color='darkorange', lw=1)
        ax.set_title('NYCTaxi — Hourly Demand Over Time')
        ax.set_ylabel('Demand')
        plt.tight_layout()
        plt.savefig('nyctaxi_temporal.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No datetime column detected in NYCTaxi — adjust column name manually.')
except:
    print('NYCTaxi not loaded. Skipping.')

## 10. Summary Table

In [ ]:
summary = []
for name, data in pems_data.items():
    T, N, F = data.shape
    summary.append({
        'Dataset': name,
        'Timesteps': T,
        'Sensors': N,
        'Features': F,
        'Duration (days)': round(T * 5 / 1440, 1),
        'Mean Flow': round(data[:, :, 0].mean(), 2),
        'Std Flow': round(data[:, :, 0].std(), 2),
        'Missing (%)': round(100 * np.isnan(data).sum() / data.size, 3),
    })

summary_df = pd.DataFrame(summary)
print('=== Dataset Summary ===' )
display(summary_df)
summary_df.to_csv('dataset_summary.csv', index=False)
print('Saved to dataset_summary.csv')